Why do we need so many layers in GX?

Great Expectations separates data management from validation. 
- The Context manages the project, 
- the Datasource defines where data comes from, 
- the Data Asset identifies the dataset, 
- the Batch Definition determines which portion of the data to validate, 
- the Batch represents the actual data, 
- theExpectations define the quality rules. 
This modular design allows the same validation rules to be reused across files, databases, and data partitions without changing the validation logic.

TOP 20 MOST COMMON GX EXPECTATIONS

1. expect_column_to_exist() → Verify required column exists.
2. expect_table_columns_to_match_set() → Verify schema columns match expected columns.
3. expect_column_values_to_not_be_null() → Check for missing values.
4. expect_column_values_to_be_null() → Verify values should be null.
5. expect_column_values_to_be_unique() → Check for duplicate values.
6. expect_compound_columns_to_be_unique() → Check composite key uniqueness.
7. expect_column_values_to_be_in_set() → Validate allowed values.
8. expect_column_values_to_not_be_in_set() → Reject invalid values.
9. expect_column_distinct_values_to_be_in_set() → Validate unique values belong to permitted list.
10. expect_column_most_common_value_to_be_in_set() → Verify dominant value is expected.
11. expect_column_values_to_be_between() → Numeric/date range validation.
12. expect_column_min_to_be_between() → Validate minimum value.
13. expect_column_max_to_be_between() → Validate maximum value.
14. expect_column_mean_to_be_between() → Validate average value.
15. expect_column_median_to_be_between() → Validate median value.
16. expect_column_values_to_match_regex() → Pattern validation using regex.
17. expect_column_values_to_not_match_regex() → Reject unwanted patterns.
18. expect_column_values_to_match_like_pattern() → SQL LIKE pattern validation.
19. expect_column_value_lengths_to_be_between() → Validate text length.
20. expect_column_values_to_be_of_type() → Validate datatype.

MOST IMPORTANT FOR DATA QUALITY ENGINEER INTERVIEWS

✔ Completeness → expect_column_values_to_not_be_null()
✔ Uniqueness → expect_column_values_to_be_unique()
✔ Validity → expect_column_values_to_be_in_set()
✔ Range Check → expect_column_values_to_be_between()
✔ Pattern Validation → expect_column_values_to_match_regex()
✔ Schema Validation → expect_column_to_exist()
✔ Datatype Validation → expect_column_values_to_be_of_type()

In [1]:
# ============================================================
# GX 1.x - Top 20 Common Expectations on sample_metadata.csv
# Version example: great_expectations==1.18.2, pandas==2.3.0
# ============================================================

import pandas as pd
import great_expectations as gx
import uuid

# ------------------------------------------------------------
# 1. Load sample metadata CSV
# ------------------------------------------------------------
df = pd.read_csv(
    r"C:\Users\mshanmugam\OneDrive - Warner Bros. Discovery\JUPYTER_PY\DQ Eng\2-PYTHON\8.Great Expectations\sample_metadata.csv"
)

# Add one column only for demonstrating "expect values to be null"
# This is useful because your original sample file does not have a fully-null column.
df["optional_release_date"] = None

# ------------------------------------------------------------
# 2. Create GX Context
# ------------------------------------------------------------
# Context is the entry point / brain of GX.
# It manages datasources, assets, expectations, validations, checkpoints, and results.
context = gx.get_context()

# ------------------------------------------------------------
# 3. Create Pandas Datasource
# ------------------------------------------------------------
# Datasource tells GX where the data comes from.
# In this case, the source is a Pandas DataFrame.
# uuid is used so this cell can be re-run without "name already exists" errors.
unique_id = uuid.uuid4().hex[:8]

datasource = context.data_sources.add_pandas(
    name=f"metadata_pandas_source_{unique_id}"
)

# ------------------------------------------------------------
# 4. Create Data Asset
# ------------------------------------------------------------
# Data Asset represents a logical dataset inside the datasource.
# Example: metadata_asset, customer_data, title_master, etc.
asset = datasource.add_dataframe_asset(
    name=f"metadata_asset_{unique_id}"
)

# ------------------------------------------------------------
# 5. Create Batch Definition
# ------------------------------------------------------------
# Batch Definition tells GX how to group the data for validation.
# Here, the full DataFrame is validated as one batch.
batch_definition = asset.add_batch_definition_whole_dataframe(
    name=f"metadata_full_batch_{unique_id}"
)

# ------------------------------------------------------------
# 6. Get Batch
# ------------------------------------------------------------
# Batch is the actual data instance that GX will validate.
batch = batch_definition.get_batch(
    batch_parameters={"dataframe": df}
)

# ------------------------------------------------------------
# 7. Create Top 20 GX 1.x Expectations
# ------------------------------------------------------------
# These are real GX expectation objects.
# They are not Pandas if/else checks.

expected_columns = [
    "series_id",
    "title",
    "season_number",
    "episode_number",
    "language_code",
    "territory",
    "synopsis",
    "optional_release_date"
]

expectations = [

    # 1. Column should exist
    gx.expectations.ExpectColumnToExist(
        column="title"
    ),

    # 2. Table columns should match expected set
    gx.expectations.ExpectTableColumnsToMatchSet(
        column_set=expected_columns,
        exact_match=True
    ),

    # 3. Column values should not be null
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="title"
    ),

    # 4. Column values should be null
    gx.expectations.ExpectColumnValuesToBeNull(
        column="optional_release_date"
    ),

    # 5. Column values should be unique
    gx.expectations.ExpectColumnValuesToBeUnique(
        column="series_id"
    ),

    # 6. Combination of columns should be unique
    gx.expectations.ExpectCompoundColumnsToBeUnique(
        column_list=["series_id", "language_code"]
    ),

    # 7. Column values should be in allowed set
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="territory",
        value_set=["US", "GB", "IN"]
    ),

    # 8. Column values should not be in blocked set
    gx.expectations.ExpectColumnValuesToNotBeInSet(
        column="territory",
        value_set=["UNKNOWN", "NA", "XYZ"]
    ),

    # 9. Distinct values should be in allowed set
    gx.expectations.ExpectColumnDistinctValuesToBeInSet(
        column="territory",
        value_set=["US", "GB", "IN"]
    ),

    # 10. Most common value should be in expected set
    gx.expectations.ExpectColumnMostCommonValueToBeInSet(
        column="territory",
        value_set=["US", "GB"]
    ),

    # 11. Values should be between range
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="season_number",
        min_value=1,
        max_value=100
    ),

    # 12. Minimum value should be between range
    gx.expectations.ExpectColumnMinToBeBetween(
        column="season_number",
        min_value=1,
        max_value=1
    ),

    # 13. Maximum value should be between range
    gx.expectations.ExpectColumnMaxToBeBetween(
        column="season_number",
        min_value=1,
        max_value=100
    ),

    # 14. Mean value should be between range
    gx.expectations.ExpectColumnMeanToBeBetween(
        column="season_number",
        min_value=1,
        max_value=100
    ),

    # 15. Median value should be between range
    gx.expectations.ExpectColumnMedianToBeBetween(
        column="season_number",
        min_value=1,
        max_value=100
    ),

    # 16. Values should match regex pattern
    # Example: en-US, en-GB
    gx.expectations.ExpectColumnValuesToMatchRegex(
        column="language_code",
        regex=r"^[a-z]{2}-[A-Z]{2}$"
    ),

    # 17. Values should not match unwanted regex pattern
    # This blocks special characters in title.
    gx.expectations.ExpectColumnValuesToNotMatchRegex(
        column="title",
        regex=r"[@#$%^&*]"
    ),

    # 18. Values should match SQL LIKE pattern
    # Example: language_code should start with en-
    gx.expectations.ExpectColumnValuesToMatchLikePattern(
        column="language_code",
        like_pattern="en-%"
    ),

    # 19. Text length should be between range
    gx.expectations.ExpectColumnValueLengthsToBeBetween(
        column="synopsis",
        min_value=10,
        max_value=500
    ),

    # 20. Values should be of expected datatype
    # For Pandas integer columns, int64 is common.
    gx.expectations.ExpectColumnValuesToBeOfType(
        column="season_number",
        type_="int64"
    )
]

# ------------------------------------------------------------
# 8. Run all GX expectations
# ------------------------------------------------------------
print("GX Version:", gx.__version__)
print("Total Rows:", len(df))
print("Total Columns:", len(df.columns))
print("=" * 80)

validation_summary = []

for i, expectation in enumerate(expectations, start=1):
    result = batch.validate(expectation)

    expectation_name = expectation.__class__.__name__
    success_status = result.success

    validation_summary.append(
        {
            "No": i,
            "Expectation": expectation_name,
            "Success": success_status
        }
    )

    print(f"{i}. {expectation_name}")
    print(f"   Success: {success_status}")

    # Print useful result details if available
    if hasattr(result, "result"):
        result_details = result.result

        if "unexpected_count" in result_details:
            print(f"   Unexpected Count: {result_details.get('unexpected_count')}")

        if "unexpected_percent" in result_details:
            print(f"   Unexpected Percent: {result_details.get('unexpected_percent')}")

        if "partial_unexpected_list" in result_details:
            print(f"   Sample Unexpected Values: {result_details.get('partial_unexpected_list')}")

    print("-" * 80)

# ------------------------------------------------------------
# 9. Final Summary as DataFrame
# ------------------------------------------------------------
summary_df = pd.DataFrame(validation_summary)

print("\nFINAL VALIDATION SUMMARY")
display(summary_df)

print("\nPassed Expectations:", summary_df["Success"].sum())
print("Failed Expectations:", len(summary_df) - summary_df["Success"].sum())

GX Version: 1.18.2
Total Rows: 5
Total Columns: 8


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

1. ExpectColumnToExist
   Success: True
--------------------------------------------------------------------------------


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

2. ExpectTableColumnsToMatchSet
   Success: True
--------------------------------------------------------------------------------


...

19. ExpectColumnValueLengthsToBeBetween
   Success: True
   Unexpected Count: 0
   Unexpected Percent: 0.0
   Sample Unexpected Values: []
--------------------------------------------------------------------------------


Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

20. ExpectColumnValuesToBeOfType
   Success: True
--------------------------------------------------------------------------------

FINAL VALIDATION SUMMARY


,No,Expectation,Success
0,1,ExpectColumnToExist,True
1,2,ExpectTableColumnsToMatchSet,True
2,3,ExpectColumnValuesToNotBeNull,True
3,4,ExpectColumnValuesToBeNull,True
4,5,ExpectColumnValuesToBeUnique,True
5,6,ExpectCompoundColumnsToBeUnique,True
6,7,ExpectColumnValuesToBeInSet,True
7,8,ExpectColumnValuesToNotBeInSet,True
8,9,ExpectColumnDistinctValuesToBeInSet,True
9,10,ExpectColumnMostCommonValueToBeInSet,True



Passed Expectations: 19
Failed Expectations: 1


## Connect GX to Snowflake — this is the real-world setup. Follow the Snowflake connector docs

In [5]:
import great_expectations as gx

context = gx.get_context()

datasource = context.data_sources.add_snowflake(
    name="ZB55325",
    account="YXIFNZY-ZB55325",
    user="MUVEESHSHANMUGAM23",
    password="MUVEE@23devamanohari",
    database="MICRO_PART_LEARN",
    schema="MP_DEMO",
    warehouse="COMPUTE_WH",
    role="ACCOUNTADMIN"
)

asset = datasource.add_table_asset(
    name="sales_asset",
    table_name="SALES_DQ_DEMO"
)

batch_definition = asset.add_batch_definition_whole_table(
    "full_table"
)

batch = batch_definition.get_batch()

expectations = [

    # Primary Key
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="SALE_ID"
    ),

    gx.expectations.ExpectColumnValuesToBeUnique(
        column="SALE_ID"
    ),

    # Region Validation
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="REGION",
        value_set=[
            "CENTRAL",
            "WEST",
            "EAST",
            "NORTH",
            "SOUTH"
        ]
    ),

    # Sales Amount Checks
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="SALES_AMOUNT"
    ),

    gx.expectations.ExpectColumnValuesToBeBetween(
        column="SALES_AMOUNT",
        min_value=0,
        max_value=1000000
    ),

    # Quantity Checks
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="QUANTITY",
        min_value=1,
        max_value=10000
    ),

    # Date Checks
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="SALE_DATE"
    ),

    # Table Volume
    gx.expectations.ExpectTableRowCountToBeBetween(
        min_value=1
    )
]

results = []

Validation=["Primary Key: sale_id","Region Validation: Region", "Sales Amount Checks: Sales Amount", "Quantity Checks: Quantity", "Date Checks: Sale Date", "Table Volume: Row Count"]
for v,exp in zip(Validation,expectations):
    result = batch.validate(exp)

    results.append(
        {
            "validation": v,
            "expectation": exp.__class__.__name__,
            "success": result.success
        }
    )

print("\nVALIDATION SUMMARY")
print("=" * 60)

for r in results:
    print(
        f"{r['validation']} - "
        f"{r['expectation']} : "
        f"{'PASS' if r['success'] else 'FAIL'}"
    )

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/12 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/13 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/13 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/13 [00:00<?, ?it/s]


VALIDATION SUMMARY
Primary Key: sale_id - ExpectColumnValuesToNotBeNull : PASS
Region Validation: Region - ExpectColumnValuesToBeUnique : FAIL
Sales Amount Checks: Sales Amount - ExpectColumnValuesToBeInSet : FAIL
Quantity Checks: Quantity - ExpectColumnValuesToNotBeNull : PASS
Date Checks: Sale Date - ExpectColumnValuesToBeBetween : FAIL
Table Volume: Row Count - ExpectColumnValuesToBeBetween : FAIL


## Create an Expectation Suite for your WBD-style metadata (title, IP type, completeness)

In [ ]:
import great_expectations as gx
import pandas as pd

connection_string = (
    "snowflake://MUVEESHKUMAR.SHANMUGAM@WBD.COM:@WBD-COMMONDATAPROD/"
    "BOLT_MSC_CDS_PROD/ATOM_BI"
    "?warehouse=CLOUD_DCP_MSC_CDS_ENGINEER_GENERAL"
    "&role=PUBLIC"
    "&authenticator=externalbrowser"
)

context = gx.get_context()

datasource = context.data_sources.add_sql(
    name="snowflake_ds",
    connection_string=connection_string,
)

asset = datasource.add_table_asset(
    name="Atom_table",
    table_name="D_TITLE"
)

batch_definition = asset.add_batch_definition_whole_table(
    "full_table"
)


# Batch is the actual data instance that GX will validate.
batch = batch_definition.get_batch()


# These are real GX expectation objects.
# They are not Pandas if/else checks.

expected_columns = [
    "NODE_IDENTIFIER",
    "AKA_PKA_TITLES",
    "ALEPH_ID",
    "ALTERNATE_IDENTIFIERS_VALUE",
    "COUNTRY_OF_ORIGIN",
    "DASH_TITLE_ID",
    "END_YEAR",
    "EPISODE_NUMBER_IN_SERIES",
    "EPISODE_PRODUCTION_NUMBER",
    "GENRE",
    "HBO_ID",
    "IBROADCAST_APAC_ID",
    "IBROADCAST_EMEA_ID",
    "IP_TYPE",
    "TITLE",
    "LIBRARY_TITLE_FULL",
    "LIBRARY_TITLE_SHORT",
    "META_ID",
    "META_INTERNATIONAL_ID",
    "MMS3_MCODE",
    "MPM_FAMILY_NUMBER",
    "MPM_NUMBER",
    "MPM_PRODUCT_NUMBER",
    "ORIGINAL_LANGUAGE",
    "ORIGINAL_MADE_FOR",
    "ORIGINAL_RELEASE_YEAR",
    "ORIGINALLY_AIRED_AS",
    "PI_UUID",
    "PROPERTY_ID"]

expectations = [

    # 1. Column should exist
    gx.expectations.ExpectColumnToExist(
        column="TITLE"
    ),

    # 2. Table columns should match expected set
    gx.expectations.ExpectTableColumnsToMatchSet(
        column_set=expected_columns,
        exact_match=True
    ),

    # 3. Column values should not be null
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="TITLE"
    ),

    # 4. Column values should be null
    gx.expectations.ExpectColumnValuesToBeNull(
        column="ALTERNATE_IDENTIFIERS_VALUE"
    ),

    # 5. Column values should be unique
    gx.expectations.ExpectColumnValuesToBeUnique(
        column="NODE_IDENTIFIER"
    ),

    # 6. Combination of columns should be unique
    gx.expectations.ExpectCompoundColumnsToBeUnique(
        column_list=["NODE_IDENTIFIER", "IP_TYPE"]
    ),

    # 7. Column values should be in allowed set
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="ORIGINAL_LANGUAGE",
        value_set=["English", "Malay"]
    ),

    # 8. Column values should not be in blocked set
    gx.expectations.ExpectColumnValuesToNotBeInSet(
        column="ORIGINAL_LANGUAGE",
        value_set=["Arabic"]
    ),

    # 9. Distinct values should be in allowed set
    gx.expectations.ExpectColumnDistinctValuesToBeInSet(
        column="ORIGINAL_LANGUAGE",
        value_set=["English", "Malay"]
    ),

    # 10. Most common value should be in expected set
    gx.expectations.ExpectColumnMostCommonValueToBeInSet(
        column="ORIGINAL_LANGUAGE",
        value_set=["English", "Malay"]
    ),

    # 11. Values should be between range
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="season_number",
        min_value=1,
        max_value=20
    ),

    # 12. Minimum value should be between range
    gx.expectations.ExpectColumnMinToBeBetween(
        column="season_number",
        min_value=1,
        max_value=1
    ),

    # 13. Maximum value should be between range
    gx.expectations.ExpectColumnMaxToBeBetween(
        column="season_number",
        min_value=1,
        max_value=20
    ),

    # 14. Mean value should be between range
    gx.expectations.ExpectColumnMeanToBeBetween(
        column="season_number",
        min_value=1,
        max_value=20
    ),

    # 15. Median value should be between range
    gx.expectations.ExpectColumnMedianToBeBetween(
        column="season_number",
        min_value=1,
        max_value=20
    ),

    # 16. Values should match regex pattern
    # Example: en-US, en-GB
    gx.expectations.ExpectColumnValuesToMatchRegex(
        column="LIBRARY_TITLE_SHORT",
        regex=r"^[a-z]{2}-[A-Z]{2}$"
    ),

    # 17. Values should not match unwanted regex pattern
    # This blocks special characters in title.
    gx.expectations.ExpectColumnValuesToNotMatchRegex(
        column="TITLE",
        regex=r"[@#$%^&*]"
    ),

    # 18. Values should match SQL LIKE pattern
    # Example: language_code should start with en-
    gx.expectations.ExpectColumnValuesToMatchLikePattern(
        column="ORIGINAL_LANGUAGE",
        like_pattern="en-%"
    ),

    # 19. Text length should be between range
    gx.expectations.ExpectColumnValueLengthsToBeBetween(
        column="LIBRARY_TITLE_FULL",
        min_value=10,
        max_value=120
    ),

    # 20. Values should be of expected datatype
    # For Pandas integer columns, int64 is common.
    gx.expectations.ExpectColumnValuesToBeOfType(
        column="season_number",
        type_="int64"
    )
]

# ------------------------------------------------------------
# 8. Run all GX expectations
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("GREAT EXPECTATIONS - SNOWFLAKE DATASET")
print("=" * 80)

print(f"GX Version      : {gx.__version__}")
print(f"Table           :  {asset.table_name}")
print("=" * 80)

validation_summary = []

for i, expectation in enumerate(expectations, start=1):
    result = batch.validate(expectation)

    expectation_name = expectation.__class__.__name__
    success_status = result.success

    validation_summary.append(
        {
            "No": i,
            "Expectation": expectation_name,
            "Success": success_status
        }
    )

    print(f"{i}. {expectation_name}")
    print(f"   Success: {success_status}")

    # Print useful result details if available
    if hasattr(result, "result"):
        result_details = result.result

        if "unexpected_count" in result_details:
            print(f"   Unexpected Count: {result_details.get('unexpected_count')}")

        if "unexpected_percent" in result_details:
            print(f"   Unexpected Percent: {result_details.get('unexpected_percent')}")

        if "partial_unexpected_list" in result_details:
            print(f"   Sample Unexpected Values: {result_details.get('partial_unexpected_list')}")

    print("-" * 80)

# ------------------------------------------------------------
# 9. Final Summary as DataFrame
# ------------------------------------------------------------
summary_df = pd.DataFrame(validation_summary)

print("\nFINAL VALIDATION SUMMARY")
display(summary_df)

print("\nPassed Expectations:", summary_df["Success"].sum())
print("Failed Expectations:", len(summary_df) - summary_df["Success"].sum())

 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://sso.wbd.com/app/snowflake/exkun8q7gmVUtforO0x7/sso/saml?SAMLRequest=lZJfb9owFMW%2FSuQ9J3aAhtYCKlaEhsoKgtB1ezPJhXoktvF1Gvj2c%2FgzdQ%2BttLfIOce%2F43tu7%2F5QFsEbWJRa9UkcMRKAynQu1bZPVuk4vCUBOqFyUWgFfXIEJPeDHoqyMHxYuVe1gH0F6AJ%2FkULe%2FOiTyiquBUrkSpSA3GV8Ofw%2B5a2IcYEI1nkcuVhylJ716pzhlNZ1HdXtSNstbTHGKLujXtVIvpB3CPM5w1jtdKaLq%2BXg3%2FQBIqas0yC8whPmF%2BNXqc4j%2BIyyPouQf0vTeTifLVMSDK%2Bve9AKqxLsEuybzGC1mJ4DoE%2FwYzJsJyxJogpDEOjCOEKl600hdpDp0lTOXxv5L7qBnBZ6K%2F2wJqM%2BMTuZs1kmoJOZ%2Bq6GWuy2JpE%2FF%2FHjzW%2B7P76MH1%2FWpZkK096l7V1Ggudrta2m2gliBRPVFOr8EWslIeuGLElZwlmH33SiOO78IsHIFyqVcCfnNTWijup1fsoljKF%2FI1M47Cp1u%2B9uy%2BeV22g7Y4duI6dNVeS8LfzEtoP%2FnkGPvrdfNu%2FJlzEZzXUhs2Mw1rYU7uOu4ig%2Bncg83JykHEohi2GeW0D0nRWFrh8sCOcX3NkKCB2cqf%2Bu%2BOAP&RelayState=ver%3A3-hint%3A22419604896493590-ETMsDgAAAZ82B5PiABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEPqalz7CpAOAgO1U9bnPNF0AAACgp0XqySSmSwVWUIVsL0c6K

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://sso.wbd.com/app/snowflake/exkun8q7gmVUtforO0x7/sso/saml?SAMLRequest=lZJRb9owFIX%2FSuQ9JzZ0DdQCKlZGl5YBKqHb%2BmYSh1o4dvB1SLpfP4fA1D60Ut8i5xx%2Fx%2FfcwXWdS%2B%2FADQithqgTEORxlehUqO0QreOp30ceWKZSJrXiQ%2FTCAV2PBsByWdBxaZ%2FVA9%2BXHKznLlJAmx9DVBpFNQMBVLGcA7UJXY1%2Fzmg3IJQBcGMdDp0sKQjHera2oBhXVRVUF4E2W9wlhGByhZ2qkXxBrxDFx4zCaKsTLc%2BW2r3pHUQHk68NwikcYXkyfhOqHcFHlE0rAvojjpf%2BcrGKkTc%2Bv%2B5GKyhzblbcHETC1w%2BzNgC4BL%2Bi8UVIwjAowecMrN8JQOkqk2zHE50XpXXXBu4LZzzFUm%2BFG1Y0GaJiJ9L4cj0L6%2Fh2l%2ByzO2Lv7Ob2%2FrecLJbTfe%2F73%2FkhZbM%2FTzWL5jtIkPd4rrbbVBsBlDxSTaHWHZFu6JOeT8KYhJRcUtINev2rJ%2BRNXKFCMXt0nlMD6KDapMdcrCjw%2F8iY17tS9fe9bf64tpk2C1L3GjluqkLtttAj24w%2BPYMBfm0%2Fbd7clRFNllqK5MWbapMz%2B35XnaBzPBGpnx2llOdMyHGaGg7gOpNSVzeGM%2BsW3JqSIzxqqW9XfPQP&RelayState=ver%3A3-hint%3A22419604896493590-ETMsDgAAAZ82B7XFABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEDT%2BshVa9zJEVk3Nxs4TfF8AAACgQJHy9OqbhZLYxPx

...

Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://sso.wbd.com/app/snowflake/exkun8q7gmVUtforO0x7/sso/saml?SAMLRequest=lZJRb9owFIX%2FSuQ9JzawhtYiVAHWLRNtUQlM6pubOOCR2MHXbtJ%2FPyfA1D200t4i5xx%2Fx%2FfcyW1bld4r1yCUjNAgIMjjMlO5kLsIbdI7%2Fxp5YJjMWakkj9AbB3Q7nQCryprG1uzlEz9aDsZzF0mg3Y8IWS2pYiCASlZxoCaj6%2Fh%2BSYcBoQyAa%2BNw6GzJQTjW3piaYtw0TdCMAqV3eEgIweQGO1Un%2BYLeIerPGbVWRmWqvFha96YPEANMvnYIp3CE1dk4E%2FI0gs8oLycR0B9puvJXj%2BsUefHldXMlwVZcr7l%2BFRnfPC1PAcAl%2BJXEo5CEYWDB5wyMPwhAqqYo2YFnqqqtcdcG7gsXPMel2gk3rGQRofog8u8zyyorlkV7%2F1sN5Xh7YHv1zG6sXI30t%2B1C%2Ftwu58nyOHuADHnbS7XDrtoEwPJEdoUad0SGoU%2FGPglTElIypleuf3L1jLyFK1RIZnrnJTWACpqXvM%2FF6hr%2FjYx5e7Dy%2BjjeVduNKZR%2BJO24k%2BOuKnTaFtqz9fS%2FZzDB7%2B3nzXtwZSSLlSpF9ubdKV0x83FXg2DQn4jcL3op5RUTZZznmgO4zspSNXPNmXELbrTlCE9P1H9XfPoH&RelayState=ver%3A3-hint%3A22419604896493590-ETMsDgAAAZ82CkbdABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAENv9iJNk6pm%2FQtTBoiS0eFAAAACgjdQ4eYLaWjutXTRzPzcRgsGuGSyE5

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://sso.wbd.com/app/snowflake/exkun8q7gmVUtforO0x7/sso/saml?SAMLRequest=lZJLb%2BIwFIX%2FSuRZJ3YABbCAKhRVpKJTpjxadWcSAxaJnfraTeivH4fHqLNope4i5xx%2Fx%2FfcwU1d5N471yCUHKIwIMjjMlWZkLshWi3v%2FB7ywDCZsVxJPkRHDuhmNABW5CWNrdnLJ%2F5mORjPXSSBNj%2BGyGpJFQMBVLKCAzUpXcQPM9oKCGUAXBuHQxdLBsKx9saUFOOqqoKqHSi9wy1CCCZ97FSN5Bf6hCi%2FZ5RaGZWq%2FGqp3Zu%2BQISYdBqEUzjC%2FGIcC3kewXeUzVkEdLpczv3542KJvPj6ulslwRZcL7h%2BFylfPc3OAcAleE7idkSiKLDgcwbGDwOQqtrm7MBTVZTWuGsD94W3PMO52gk3rGQyROVBZILPX5I%2B%2F7i30%2Bc%2FoZptFzZeH1%2Fb9UbU9zAti70Yk4cuG8edFHnra7WtptoEwPJENoUad0RakU%2B6PomWJKKkR0kniPrkFXkTV6iQzJyc19QAKqg22SkXK0v8LzLm9cHK3lt3V6xXZqv0I6m7jRw3VaHzttATW49%2BPIMB%2Fmy%2FbN5vV0YymatcpEfvTumCma%2B7CoPwdCIyf3uSUl4wkcdZpjmA6yzPVXWrOTNuwY22HOHRmfr%2Fio%2F%2BAg%3D%3D&RelayState=ver%3A3-hint%3A22419604896493590-ETMsDgAAAZ82CnxSABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAELtUiCQ8UPN1tMMQ5QG1nokAAACgx83TsYzVFIfZKvKivfPSL

NameError: name 'pd' is not defined

**Run a Checkpoint & Generate Data Docs:**

• Connect Great Expectations to a Snowflake datasource.  
• Create a Data Asset and Batch Definition for the Snowflake table.  
• Create an Expectation Suite containing all data quality rules (null checks, uniqueness checks, range validations, domain validations, row count checks, etc.).  
• Create a Validation Definition to link the Snowflake data with the Expectation Suite.  
• Create a Checkpoint as a reusable production validation object.  
• Run the Checkpoint to execute all expectations against actual Snowflake data and generate validation results (PASS/FAIL).  
• Build Data Docs to generate an HTML Data Quality report.  
• Use Data Docs to review validation status, passed and failed expectations, unexpected values, validation statistics, and overall data quality health.  
• In production, Checkpoints are typically executed from Airflow, ADF, Snowflake Tasks, or other orchestration tools to automate data quality monitoring and reporting. [\[docs.great...tations.io\]](https://docs.greatexpectations.io/docs/reference/api/checkpoint_class/), [\[docs.great...tations.io\]](https://docs.greatexpectations.io/docs/0.18/reference/learn/terms/checkpoint/)



✅ Expectation Suite

A collection of data quality rules (Expectations).
Does not validate data by itself.
Simply stores all the expectations you want to enforce on a dataset.

✅ Batch

Represents the actual data to be validated (e.g., Snowflake table SALES_DQ_DEMO).

✅ Validation Definition

Connects the Expectation Suite with the Batch.
Defines:"Validate this data using these rules."

✅ Checkpoint

Executes the Validation Definition.
Runs all expectations in the suite against the Snowflake batch.
Produces validation results (PASS/FAIL).

✅ Data Docs

Generates an HTML report from the validation results.
Shows:

- Passed Expectations ✅
- Failed Expectations ❌
- Unexpected Values
- Success Percentage
- Validation Statistics
- Data Quality Summary


WORK FLOW:

- Snowflake Table (Batch)
            +
- Expectation Suite
            ↓
- Validation Definition
            ↓
- Checkpoint.run()
            ↓
- Validation Results
            ↓
- Data Docs HTML Report


In [1]:
import great_expectations as gx

# -----------------------------------------------------------------------------
# CONTEXT
# -----------------------------------------------------------------------------

context = gx.get_context()

# -----------------------------------------------------------------------------
# DATASOURCE
# -----------------------------------------------------------------------------

datasource = context.data_sources.add_snowflake(
    name="ZB55325",
    account="YXIFNZY-ZB55325",
    user="MUVEESHSHANMUGAM23",
    password="MUVEE@23devamanohari",
    database="MICRO_PART_LEARN",
    schema="MP_DEMO",
    warehouse="COMPUTE_WH",
    role="ACCOUNTADMIN"
)

# -----------------------------------------------------------------------------
# TABLE ASSET
# -----------------------------------------------------------------------------

asset = datasource.add_table_asset(
    name="sales_asset",
    table_name="SALES_DQ_DEMO"
)

batch_definition = asset.add_batch_definition_whole_table(
    "full_table"
)

batch = batch_definition.get_batch()

# -----------------------------------------------------------------------------
# EXPECTATION SUITE
# -----------------------------------------------------------------------------

suite = gx.ExpectationSuite(
    name="sales_dq_suite"
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="SALE_ID"
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeUnique(
        column="SALE_ID"
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="REGION",
        value_set=[
            "CENTRAL",
            "WEST",
            "EAST",
            "NORTH",
            "SOUTH"
        ]
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="SALES_AMOUNT"
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="SALES_AMOUNT",
        min_value=0,
        max_value=1000000
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="QUANTITY",
        min_value=1,
        max_value=10000
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="SALE_DATE"
    )
)

suite.add_expectation(
    gx.expectations.ExpectTableRowCountToBeBetween(
        min_value=1
    )
)

context.suites.add(suite)

# -----------------------------------------------------------------------------
# VALIDATION DEFINITION
# -----------------------------------------------------------------------------

validation_definition = gx.ValidationDefinition(
    name="sales_validation",
    data=batch_definition,
    suite=suite
)

context.validation_definitions.add(
    validation_definition
)

# -----------------------------------------------------------------------------
# CHECKPOINT
# -----------------------------------------------------------------------------

checkpoint = gx.Checkpoint(
    name="sales_checkpoint",
    validation_definitions=[
        validation_definition
    ]
)

context.checkpoints.add(checkpoint)

# -----------------------------------------------------------------------------
# RUN CHECKPOINT
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("RUNNING CHECKPOINT")
print("=" * 80)

checkpoint_result = checkpoint.run()

print(
    f"Validation Status : "
    f"{'PASS' if checkpoint_result.success else 'FAIL'}"
)

# -----------------------------------------------------------------------------
# DATA DOCS
# -----------------------------------------------------------------------------

print("\nBuilding Data Docs...")

context.build_data_docs()

print("\nDATA DOCS GENERATED SUCCESSFULLY")

for site in context.get_docs_sites_urls():
    print(
        f"Site Name : {site['site_name']}"
    )
    print(
        f"Site URL  : {site['site_url']}"
    )

print("\n" + "=" * 80)
print("CHECKPOINT COMPLETED")
print("=" * 80)


RUNNING CHECKPOINT


Calculating Metrics:   0%|          | 0/55 [00:00<?, ?it/s]

Validation Status : FAIL

Building Data Docs...

DATA DOCS GENERATED SUCCESSFULLY
Site Name : local_site
Site URL  : file://C:\Users\MSHANM~1\AppData\Local\Temp\tmpy7o5c6h9\index.html

CHECKPOINT COMPLETED
